In [2]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import make_column_selector as selector, ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [3]:
p = Path.cwd()
data_path = p / '..' / 'data' / 'Loan_approval_data_2025.csv'
data = pd.read_csv(data_path, header=0)

In [ ]:
def load_data(file_name='Loan_approval_data_2025.csv'):
    data = pd.read_csv(data / file_name, header=0)
    # Reduce memory pressure by using narrower dtypes where possible.
    float_cols = data.select_dtypes(include=['float64']).columns
    int_cols = data.select_dtypes(include=['int64']).columns
    string_cols = data.select_dtypes(include=['string', 'str']).columns
    object_cols = data.columns[data.dtypes == object]

    for col in float_cols:
        data[col] = pd.to_numeric(data[col], downcast='float')
    for col in int_cols:
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in set(string_cols).union(set(object_cols)):
        data[col] = data[col].astype('category')

    return data

In [ ]:
def feature_engineernig(data):
    y = data['loan_status']
    # Recreate full feature matrix for this experiment
    X_full = data.drop(['loan_status', 'customer_id'], axis=1)

    # Light feature engineering based on ratios and risk interactions
    X_full = X_full.copy()
    X_full['income_after_debt'] = X_full['annual_income'] - X_full['current_debt']
    X_full['debt_x_dti'] = X_full['current_debt'] * X_full['debt_to_income_ratio']
    X_full['rate_x_lti'] = X_full['interest_rate'] * X_full['loan_to_income_ratio']

    return X_full, y

In [ ]:
def preprocess_data(X, y):
    Xf_train, Xf_test, yf_train, yf_test = train_test_split(
        X, y, test_size=0.2, random_state=3, stratify=y
    )

    # Select numeric columns directly; treat all remaining columns as categorical.
    numeric_features = Xf_train.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [col for col in Xf_train.columns if col not in numeric_features]

    numeric_preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32))
    ])

    preprocessor_full = ColumnTransformer(
        transformers=[
            ('num', numeric_preprocessor, numeric_features),
            ('cat', categorical_preprocessor, categorical_features),
        ]
    )

    return {
        "preprocessor_full": preprocessor_full,
        "Xf_train": Xf_train,
        "Xf_test": Xf_test,
        "yf_train": yf_train,
        "yf_test": yf_test
    }

In [ ]:
def train_test_classifier(preprocessor_full, Xf_train, Xf_test, yf_train, yf_test):
    # Logistic Regression with richer preprocessing
    clf_lr_full = Pipeline(
        steps=[
            ('preprocessor', preprocessor_full),
            ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced'))
        ]
    )

    clf_lr_full.fit(Xf_train, yf_train)
    pred_lr = clf_lr_full.predict(Xf_test)
    proba_lr = clf_lr_full.predict_proba(Xf_test)[:, 1]

    print(f"Full LR accuracy: {accuracy_score(yf_test, pred_lr):.4f}")
    print(f"Full LR ROC-AUC: {roc_auc_score(yf_test, proba_lr):.4f}")
    print(classification_report(yf_test, pred_lr))

    # SVC with richer preprocessing
    clf_svc_full = Pipeline(
        steps=[
            ('preprocessor', preprocessor_full),
            # Keep probability=False to avoid costly internal calibration that increases memory usage.
            ('classifier', SVC(kernel='rbf', C=2.0, gamma='scale', probability=False, class_weight='balanced'))
        ]
    )

    clf_svc_full.fit(Xf_train, yf_train)
    pred_svc = clf_svc_full.predict(Xf_test)
    score_svc = clf_svc_full.decision_function(Xf_test)

    print(f"Full SVC accuracy: {accuracy_score(yf_test, pred_svc):.4f}")
    print(f"Full SVC ROC-AUC: {roc_auc_score(yf_test, score_svc):.4f}")
    print(classification_report(yf_test, pred_svc))

    # XGBoost with non-linear classification capabilities
    clf_xgb_full = Pipeline(
        steps=[
            ('preprocessor', preprocessor_full),
            ('classifier', XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, colsample_bytree=0.8, subsample=0.8, eval_metric='logloss', objective='binary:logistic', tree_method='hist', n_jobs=-1, random_state=42))
        ]
    )

    clf_xgb_full.fit(Xf_train, yf_train)
    pred_xgb = clf_xgb_full.predict(Xf_test)
    proba_xgb = clf_xgb_full.predict_proba(Xf_test)[:, 1]

    print(f"Full XGB accuracy: {accuracy_score(yf_test, pred_xgb):.4f}")
    print(f"Full XGB ROC-AUC: {roc_auc_score(yf_test, proba_xgb):.4f}")
    print(classification_report(yf_test, pred_xgb))

    model_predictions = pd.DataFrame(
        {
            'source_index': yf_test.index.to_numpy(),
            'y_true': yf_test.to_numpy(dtype=np.int8),
            'pred_lr': np.asarray(pred_lr, dtype=np.int8),
            'proba_lr': np.asarray(proba_lr, dtype=np.float16),
            'pred_svc': np.asarray(pred_svc, dtype=np.int8),
            'proba_svc': np.asarray(score_svc, dtype=np.float16),
            'pred_xgb': np.asarray(pred_xgb, dtype=np.int8),
            'proba_xgb': np.asarray(proba_xgb, dtype=np.float16)
        }, 
        ).sort_values('source_index').reset_index(drop=True)
    
    model_predictions['lr_correct'] = model_predictions['pred_lr'] == model_predictions['y_true']
    model_predictions['svc_correct'] = model_predictions['pred_svc'] == model_predictions['y_true']
    model_predictions['xgb_correct'] = model_predictions['pred_xgb'] == model_predictions['y_true']

    model_predictions.to_csv(p / '..' / 'data' / 'predictions.csv', index=False)

In [ ]:
data = load_data()
X, y = feature_engineernig(data=data)
bundle = preprocess_data(X, y)
train_test_classifier(**bundle)

In [ ]:
# Stats
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

p = Path().cwd() / '..' / 'data'

model_predictions = pd.read_csv(p / 'predictions.csv')

# McNemar test setup: contingency table based on correctness of each model

# 2x2 contingency table
# rows: LR correct?   columns: SVC correct?
cm = pd.crosstab(model_predictions['lr_correct'], model_predictions['svc_correct'])
cm = cm.reindex(index=[True, False], columns=[True, False], fill_value=0)

print('McNemar contingency table (rows=LR, cols=SVC):')
print(cm)

# Off-diagonal cells used by McNemar
b = cm.loc[True, False]   # LR correct, SVC wrong
c = cm.loc[False, True]   # LR wrong, SVC correct
print(f'b (LR only correct): {b}')
print(f'c (SVC only correct): {c}')

res = mcnemar(cm, exact=False, correction=True)
print(f"Stats: {res.statistic}")
print(f"p-value: {res.pvalue}")

# 2x2 contingency table
# rows: LR correct?   columns: XGB correct?
cm = pd.crosstab(model_predictions["lr_correct"], model_predictions["xgb_correct"])
cm.reindex(index=[True, False], columns=[True, False], fill_value=0)

print('McNemar contingency table (rows=LR, cols=XGB):')
print(cm)

# Off-diagonal cells used by McNemar
b = cm.loc[True, False]   # LR correct, XGB wrong
c = cm.loc[False, True]   # LR wrong, XGB correct
print(f'b (LR only correct): {b}')
print(f'c (XGB only correct): {c}')

res = mcnemar(cm, exact=False, correction=True)
print(f"Stats: {res.statistic}")
print(f"p-value: {res.pvalue}")

In [ ]:
# Bootstraping
METRIC_FUNCS = {
    "accuracy": accuracy_score,
    "precision": precision_score,
    "recall": recall_score,
    "f1": f1_score,
}


def bootstrap_metric_delta_ci(
    df,
    cols,
    metric="accuracy",
    n_resamples=1000,
    confidence_level=0.95,
    random_state=42,
):
    """Low-memory bootstrap CI for metric delta: LR - SVC.

    Keeps only a compact float array of bootstrap deltas and avoids
    retaining large intermediate bootstrap objects.
    """
    if metric not in METRIC_FUNCS:
        valid = ", ".join(METRIC_FUNCS)
        raise ValueError(f"Unsupported metric '{metric}'. Use one of: {valid}")

    score_fn = METRIC_FUNCS[metric]
    y_true = df["y_true"].to_numpy(dtype=np.int8)
    pred_m1 = df[cols[0]].to_numpy(dtype=np.int8)
    pred_m2 = df[cols[1]].to_numpy(dtype=np.int8)

    if not (len(y_true) == len(pred_m1) == len(pred_m2)):
        raise ValueError(f"Input columns y_true, {cols[0]}, {cols[1]} must have same length")

    def statistic(y, m1, m2):
        return score_fn(y, m2) - score_fn(y, m1)

    rng = np.random.default_rng(random_state)
    n = y_true.shape[0]
    deltas = np.empty(n_resamples, dtype=np.float32)

    for i in range(n_resamples):
        idx = rng.integers(0, n, n)
        deltas[i] = statistic(y_true[idx], pred_m1[idx], pred_m2[idx])

    alpha = 1.0 - confidence_level
    ci_low = float(np.quantile(deltas, alpha / 2.0))
    ci_high = float(np.quantile(deltas, 1.0 - alpha / 2.0))
    point_estimate = float(statistic(y_true, pred_m1, pred_m2))
    p_two_sided = float(2.0 * min((deltas <= 0).mean(), (deltas >= 0).mean()))

    return {
        "metric": metric,
        "point_estimate": point_estimate,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value_two_sided": p_two_sided,
        "n_resamples": n_resamples,
        "random_state": random_state,
    }


accuracy_delta = bootstrap_metric_delta_ci(model_predictions, cols=["pred_lr", "pred_svc"], metric="accuracy", n_resamples=10000)
print(accuracy_delta)

accuracy_delta = bootstrap_metric_delta_ci(model_predictions, cols=["pred_lr", "pred_xgb"], metric="accuracy", n_resamples=10000)
print(accuracy_delta)